# 🔍 03: Fraud Detection Model

Train, evaluate, and explain the XGBoost fraud detection model.

---

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
    f1_score, confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay)
import xgboost as xgb

sns.set_theme(style='whitegrid')
print('Ready ✓')

## 1. Load Features

In [ ]:
# Run pipeline first if needed
import subprocess
if not os.path.exists('../data/processed/features_dev.parquet'):
    print('Running pipeline...')
    subprocess.run([sys.executable, '../src/ingestion/pipeline_runner.py', '--source', 'synthetic'])

df = pd.read_parquet('../data/processed/features_dev.parquet')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["fraud_label"].mean()*100:.2f}%')
df.describe().T.style.background_gradient(cmap='Blues')

## 2. Train Model

In [ ]:
from src.models.fraud_model import load_features, temporal_split, prepare_xy, train_model

train_df, test_df = temporal_split(df, test_frac=0.20)
train_df, val_df  = temporal_split(train_df, test_frac=0.15)

X_train, y_train, feat_cols = prepare_xy(train_df)
X_val,   y_val,   _         = prepare_xy(val_df)
X_test,  y_test,  _         = prepare_xy(test_df)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Features: {len(feat_cols)}')

In [ ]:
model, imputer = train_model(X_train, y_train, X_val, y_val)
print('Training complete ✓')

## 3. Evaluation

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer

X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)
y_proba = model.predict_proba(X_test_imp)[:,1]
y_pred  = (y_proba >= 0.5).astype(int)

print(f'AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'F1:        {f1_score(y_test, y_pred):.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Legit','Pred Fraud'],
            yticklabels=['True Legit','True Fraud'], ax=ax)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC and PR curves
fig, axes = plt.subplots(1,2, figsize=(14,5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0], color='#e74c3c')
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].plot([0,1],[0,1],'k--', alpha=0.5)

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[1], color='#3498db')
axes[1].set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Importance & SHAP

In [ ]:
# XGBoost native importance
importance = pd.Series(model.feature_importances_, index=feat_cols).nlargest(20)

fig, ax = plt.subplots(figsize=(12, 7))
importance.sort_values().plot(kind='barh', ax=ax, color='#3498db', edgecolor='black')
ax.set_title('Top 20 Feature Importances (XGBoost)', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_imp.head(500))

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_imp.head(500), plot_type='bar', show=False)
    plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Beeswarm plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_imp.head(500), show=False)
    plt.title('SHAP Beeswarm Plot', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('SHAP not installed — pip install shap')

## 5. Threshold Optimization

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    yp = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, yp)
    tn,fp,fn,tp = cm_t.ravel()
    results.append({
        'threshold': t,
        'precision': precision_score(y_test,yp,zero_division=0),
        'recall':    recall_score(y_test,yp,zero_division=0),
        'f1':        f1_score(y_test,yp,zero_division=0),
        'fpr':       fp/max(fp+tn,1),
    })

res_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(res_df['threshold'], res_df['precision'], label='Precision', color='#2ecc71', lw=2)
ax.plot(res_df['threshold'], res_df['recall'],    label='Recall',    color='#e74c3c', lw=2)
ax.plot(res_df['threshold'], res_df['f1'],        label='F1',        color='#3498db', lw=2)
ax.plot(res_df['threshold'], res_df['fpr'],       label='FPR',       color='#e67e22', lw=2, linestyle='--')
ax.axhline(0.08, color='gray', linestyle=':', label='Target FPR=8%')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Metrics vs Decision Threshold', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

best_row = res_df.loc[res_df['f1'].idxmax()]
print(f'Best threshold by F1: {best_row["threshold"]:.2f}')
print(f'  Precision: {best_row["precision"]:.3f}')
print(f'  Recall:    {best_row["recall"]:.3f}')
print(f'  F1:        {best_row["f1"]:.3f}')
print(f'  FPR:       {best_row["fpr"]:.3f}')